# APS360 — Transcript Classifier: Colab Training

> **Before running:** Enable GPU via `Runtime → Change runtime type → T4 GPU`

## 1. Clone repo & install dependencies

In [1]:
import os

REPO_URL = "https://github.com/romeluis/APS360-TranscriptClassifier.git"
REPO_DIR = "/content/APS360-TranscriptClassifier"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
    print("Repo already cloned — pulled latest changes.")

%cd {REPO_DIR}

Cloning into '/content/APS360-TranscriptClassifier'...
remote: Enumerating objects: 8377, done.
remote: Counting objects: 100% (2902/2902), done.
remote: Compressing objects: 100% (463/463), done.
remote: Total 8377 (delta 2507), reused 2807 (delta 2426), pack-reused 5475 (from 2)
Receiving objects: 100% (8377/8377), 69.26 MiB | 53.01 MiB/s, done.
Resolving deltas: 100% (5062/5062), done.
/content/APS360-TranscriptClassifier


In [2]:
# Python packages
!pip install -q -r requirements.txt
!pip install -q pytorch-crf    # CRF decoder for TranscriptNERModel

# WeasyPrint system libs — required to render HTML → PDF for training data
!apt-get install -q -y libpango-1.0-0 libpangoft2-1.0-0 libharfbuzz0b 2>/dev/null

import torch, multiprocessing
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU:    {gpu_name}")
    print(f"VRAM:   {vram_gb:.1f} GB")
    # Recommend batch size based on GPU
    if vram_gb >= 35:
        print("  → A100 detected: BATCH_SIZE=48 is set in model/config.py")
    else:
        print("  → T4 detected: consider setting BATCH_SIZE=16 in model/config.py")
else:
    print("WARNING: No GPU found — training will be very slow!")
print(f"CPUs:   {multiprocessing.cpu_count()}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.8/319.8 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 55.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 111.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.4/829.4 kB 47.4 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
libharfbuzz0b is already the newest version (2.7.4-1ubuntu3.2).
libharfbuzz0b set to manually installed.
libpango-1.0-0 is already the newest version (1.50.6+ds-2ubuntu1).
libpango-1.0-0 set to manually installed.
libpangoft2-1.0-0 is already the newest version (1.50.6+ds-2ubuntu1).
libpangoft2-1.0-0 set to manually installed.
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
D

## 2. Mount Google Drive

In [4]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR  = "/content/drive/MyDrive/APS360_checkpoints"
DRIVE_ZIP  = f"{DRIVE_DIR}/output.zip"   # training data cached as a single zip
!mkdir -p {DRIVE_DIR}

Mounted at /content/drive


## 3. Generate training data (or restore from Drive)

**Pipeline:** HTML template → WeasyPrint → PDF → pypdf text → character-aligned BIO labels

- Checks Drive for `output.zip` first — if found, extracts it (~1 min) instead of regenerating (~55 min)
- After generation, zips the output folder and saves it to Drive as `output.zip`
- `--count 200`: 61 templates × 200 = **12,200 training samples**
- `--workers 2`: matches Colab's 2 CPU cores

In [5]:
import glob, os, zipfile

COUNT_PER_TEMPLATE = 200
N_TEMPLATES = len(glob.glob("transcript_generator/templates/*.html"))
EXPECTED = N_TEMPLATES * COUNT_PER_TEMPLATE
OUT_DIR = "transcript_generator/output"

def _count_local():
    return len(glob.glob(f"{OUT_DIR}/*/*.json"))

def _extract_zip():
    """Extract DRIVE_ZIP into transcript_generator/ (recreates output/ folder)."""
    print(f"Extracting {DRIVE_ZIP} ...")
    with zipfile.ZipFile(DRIVE_ZIP, "r") as zf:
        total = len(zf.namelist())
        zf.extractall("transcript_generator/")
    extracted = _count_local()
    print(f"✓ Extracted {extracted} samples from output.zip")
    return extracted

def _save_zip():
    """Zip transcript_generator/output/ and save to DRIVE_ZIP."""
    print(f"Zipping {OUT_DIR}/ → {DRIVE_ZIP} ...")
    with zipfile.ZipFile(DRIVE_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files in os.walk(OUT_DIR):
            for fname in files:
                fpath = os.path.join(root, fname)
                # Store as output/<template>/<file> (relative to transcript_generator/)
                arcname = os.path.relpath(fpath, "transcript_generator/")
                zf.write(fpath, arcname)
    size_mb = os.path.getsize(DRIVE_ZIP) / 1e6
    print(f"✓ Saved output.zip ({size_mb:.0f} MB) to Drive")

# ── Main logic ──────────────────────────────────────────────────────────────
print(f"Templates: {N_TEMPLATES}  |  Expected samples: {EXPECTED}")

existing = _count_local()
print(f"Existing local JSON files: {existing}")

if existing >= EXPECTED:
    print("✓ Data already present in runtime — skipping.")

elif os.path.isfile(DRIVE_ZIP):
    print(f"Found {DRIVE_ZIP} — restoring from zip...")
    restored = _extract_zip()
    if restored < EXPECTED:
        print(f"Zip incomplete ({restored} < {EXPECTED}) — regenerating...")
        get_ipython().system(f"""python transcript_generator/main.py \
            --all-templates \
            --count {COUNT_PER_TEMPLATE} \
            --pdf-extract \
            --augment-noise \
            --no-pdf \
            --workers 2""")
        _save_zip()

else:
    print(f"No output.zip found — generating {EXPECTED} transcripts (~55 min)...")
    get_ipython().system(f"""python transcript_generator/main.py \
        --all-templates \
        --count {COUNT_PER_TEMPLATE} \
        --pdf-extract \
        --augment-noise \
        --no-pdf \
        --workers 2""")
    _save_zip()


Templates: 59  |  Expected samples: 11800
Existing local JSON files: 0
Found /content/drive/MyDrive/APS360_checkpoints/output.zip — restoring from zip...
Extracting /content/drive/MyDrive/APS360_checkpoints/output.zip ...
✓ Extracted 11800 samples from output.zip


## 4. Train

In [6]:
import sys
sys.path.insert(0, REPO_DIR)

from model.train import train

history = train(
    data_dir="transcript_generator/output",
    output_dir="model/checkpoints",
)

Device: cuda  |  bf16: True
Loading and splitting data...
  Train: 7000 samples (35 templates)
  Val:   2400 samples (12 templates)
  Test:  2400 samples (12 templates)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]


Training for 15 epochs (3810 steps)...

  Epoch 1 step 1/254  loss=422.6389
  Epoch 1 step 5/254  loss=nan
  Epoch 1 step 10/254  loss=nan


KeyboardInterrupt: 

## 5. Plot training curves

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(epochs, history["train_loss"], "o-", label="train")
axes[0].plot(epochs, history["val_loss"],   "s-", label="val")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, history["val_token_acc"], "o-", color="royalblue")
axes[1].set_title("Val Token Accuracy"); axes[1].grid(alpha=0.3)

axes[2].plot(epochs, history["val_entity_f1"], "o-", color="green")
axes[2].set_title("Val Entity F1"); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("training_history.png", dpi=150)
plt.show()

## 6. Save checkpoint to Google Drive

In [ ]:
!rm -rf {DRIVE_DIR}/best
!cp -r model/checkpoints/best              {DRIVE_DIR}/
!cp model/checkpoints/split_info.json      {DRIVE_DIR}/ 2>/dev/null || true
!cp model/logs/history.json                {DRIVE_DIR}/ 2>/dev/null || true
!cp training_history.png                   {DRIVE_DIR}/ 2>/dev/null || true

print(f"✓ Checkpoint saved to {DRIVE_DIR}")

## 7. (Optional) Download checkpoint directly

Alternative to Drive — zip and download straight to your Mac.

In [ ]:
!zip -r bert_checkpoint.zip model/checkpoints/best model/logs/history.json training_history.png

from google.colab import files
files.download("bert_checkpoint.zip")

## 8. (Optional) Smoke test on a test-split sample

In [ ]:
import json, glob
from model.predict import BertNERPredictor
from evaluation.reconstructor import reconstruct_semesters

with open("model/checkpoints/split_info.json") as f:
    split_info = json.load(f)

test_template = split_info["test_templates"][0]
sample_path = sorted(glob.glob(f"transcript_generator/output/{test_template}/*.json"))[0]
with open(sample_path) as f:
    sample = json.load(f)

predictor = BertNERPredictor(checkpoint_dir="model/checkpoints/best")
tags = predictor.predict(sample["tokens"])
semesters = reconstruct_semesters(sample["tokens"], tags)

print(f"Template: {test_template}\n")
for sem in semesters:
    print(f"  {sem['semester_name']}")
    for c in sem["courses"]:
        print(f"    {c['code']:<12} {c['name']:<40} {c['grade']}")